In [21]:
import pandas as pd
import numpy as np

In [22]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [23]:
train_data =pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

In [24]:
train_data

,year,month,day,order,country,session_id,page1_main_category,page2_clothing_model,colour,location,model_photography,price,price_2,page
0,2008,6,22,21,29,15648,3,C20,13,1,2,48,1,2
1,2008,5,19,6,29,10018,2,B26,13,3,1,57,1,2
2,2008,7,15,2,29,19388,3,C13,9,5,1,48,1,1
3,2008,5,2,2,29,7181,2,B11,2,4,1,43,2,1
4,2008,6,9,16,29,13493,2,B31,9,5,1,57,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132374,2008,7,4,3,29,17622,4,P19,2,1,1,48,1,2
132375,2008,6,19,9,29,15165,3,C26,14,3,1,28,2,2
132376,2008,7,15,4,29,19359,1,A4,3,2,2,38,2,1
132377,2008,7,28,16,29,21454,3,C50,9,5,2,20,2,3


In [25]:
test_data

,year,month,day,order,country,session_id,page1_main_category,page2_clothing_model,colour,location,model_photography,price,price_2,page
0,2008,4,22,4,29,5279,4,P48,9,4,2,33,2,3
1,2008,5,19,1,29,10059,1,A15,14,5,2,33,2,1
2,2008,4,11,10,29,2919,4,P23,6,2,2,28,2,2
3,2008,4,28,3,27,6304,2,B24,11,2,1,57,1,2
4,2008,5,26,1,29,11266,1,A2,3,1,1,43,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33090,2008,5,13,1,29,8885,1,A5,3,2,1,43,2,1
33091,2008,4,14,3,29,3546,1,A15,14,5,2,33,2,1
33092,2008,6,13,37,29,14336,2,B9,1,3,1,48,2,1
33093,2008,5,23,16,29,10786,3,C34,7,6,1,48,1,2


In [26]:
train_data.isnull().sum()

year                    0
month                   0
day                     0
order                   0
country                 0
session_id              0
page1_main_category     0
page2_clothing_model    0
colour                  0
location                0
model_photography       0
price                   0
price_2                 0
page                    0
dtype: int64

In [27]:
from sklearn.preprocessing import OrdinalEncoder

In [28]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

In [29]:
# Encode category values in train and test with the same fitted encoder;
# unseen test labels become -1 instead of raising an error.
train_data[['page2_clothing_model']] = encoder.fit_transform(train_data[['page2_clothing_model']])
test_data[['page2_clothing_model']] = encoder.transform(test_data[['page2_clothing_model']])


In [30]:
train_data

,year,month,day,order,country,session_id,page1_main_category,page2_clothing_model,colour,location,model_photography,price,price_2,page
0,2008,6,22,21,29,15648,3,88.0,13,1,2,48,1,2
1,2008,5,19,6,29,10018,2,60.0,13,3,1,57,1,2
2,2008,7,15,2,29,19388,3,80.0,9,5,1,48,1,1
3,2008,5,2,2,29,7181,2,45.0,2,4,1,43,2,1
4,2008,6,9,16,29,13493,2,66.0,9,5,1,57,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132374,2008,7,4,3,29,17622,4,145.0,2,1,1,48,1,2
132375,2008,6,19,9,29,15165,3,94.0,14,3,1,28,2,2
132376,2008,7,15,4,29,19359,1,33.0,3,2,2,38,2,1
132377,2008,7,28,16,29,21454,3,121.0,9,5,2,20,2,3


In [31]:

from sklearn.preprocessing import StandardScaler
train_features = train_data[['page1_main_category', 'page2_clothing_model', 'colour', 'order', 'price', 'page','location', 'model_photography']]
train_target = train_data['price_2']

test_features = test_data[['page1_main_category', 'page2_clothing_model', 'colour', 'order', 'price','page', 'location', 'model_photography']]
test_target = test_data['price_2']

scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

In [33]:
model_params = {
    "Logistic_Regression": (LogisticRegression(), {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["liblinear", "lbfgs"]
    }),

    "Random_Forest": (RandomForestClassifier(), {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10]
    }),

    "Decision_Tree": (DecisionTreeClassifier(), {
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "criterion": ["gini", "entropy"]
    })
}

In [34]:
from sklearn.model_selection import RandomizedSearchCV
import time

# Use RandomizedSearchCV with fewer iterations and smaller CV to speed up hyperparameter search
reports = []
for name, (model, param_grid) in model_params.items():
    start = time.time()
    if param_grid:
        rs = RandomizedSearchCV(model, param_grid, n_iter=20, cv=3, scoring="accuracy", n_jobs=-1, random_state=42)
        rs.fit(train_features, train_target)
        best_model = rs.best_estimator_
        best_params = rs.best_params_
    else:
        best_model = model
        best_model.fit(train_features, train_target)
        best_params = "Default Parameters"

    predictions = best_model.predict(test_features)
    accuracy = accuracy_score(test_target, predictions)
    report = classification_report(test_target, predictions)
    confusion = confusion_matrix(test_target, predictions)
    duration = time.time() - start

    reports.append((name, best_model, best_params, accuracy, report, confusion, duration))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [35]:

for name, model, best_params, accuracy, report, confusion, duration in reports:
    print(f"Model: {name}")
    print(f"Best Parameters: {best_params}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Duration (s): {duration:.2f}")
    print(f"Classification Report:\n{report}")
    print(f"Confusion Matrix:\n{confusion}\n")

Model: Logistic_Regression
Best Parameters: {'solver': 'liblinear', 'C': 1}
Accuracy: 0.9984
Duration (s): 16.12
Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     16981
           2       1.00      1.00      1.00     16114

    accuracy                           1.00     33095
   macro avg       1.00      1.00      1.00     33095
weighted avg       1.00      1.00      1.00     33095

Confusion Matrix:
[[16930    51]
 [    1 16113]]

Model: Random_Forest
Best Parameters: {'n_estimators': 200, 'min_samples_split': 10, 'max_depth': None}
Accuracy: 1.0000
Duration (s): 204.97
Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     16981
           2       1.00      1.00      1.00     16114

    accuracy                           1.00     33095
   macro avg       1.00      1.00      1.00     33095
weighted avg       1.00      1.00      1.00     33

In [43]:
import pickle

with open("random_forest_classifier_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("classification_standard_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)